# 01 — Get the data

Stage 1 of 4. Downloads the **horses** and **races** files for **2010–2020** from the Kaggle dataset
[hwaitt/horse-racing](https://www.kaggle.com/datasets/hwaitt/horse-racing) and saves them, unchanged, to `data/raw/`.

**Needs a free Kaggle account and API token.** On [kaggle.com/settings/api](https://www.kaggle.com/settings/api) click *Generate New Token*, then either
set the environment variable `KAGGLE_API_TOKEN` to it, or save it in the file `~/.kaggle/access_token` (in your home folder, **outside** this repository).
Never put the token in the code or commit it.

Alternative without a token: download the files by hand from the dataset page and put them in `data/raw/`; this notebook then skips the download and only checks them.

**Input:** none  
**Output:** `data/raw/horses_<year>.csv`, `data/raw/races_<year>.csv` for 2010–2020, and `data/raw/dataset-metadata.json` (licence and source information)

In [ ]:
import sys
from pathlib import Path

# Make code/helpers.py importable whether the notebook is run from code/ or the repo root.
sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "helpers.py").exists() else Path.cwd() / "code"))

import json

import numpy as np
import pandas as pd

import helpers
from helpers import DATA_RAW, DATA_PROCESSED, FIGURES

helpers.set_seed()
helpers.ensure_dirs()

## Settings

In [ ]:
DATASET = "hwaitt/horse-racing"
YEARS = helpers.YEARS          # 2010–2020, defined once in helpers.py
KINDS = ("horses", "races")

wanted = [f"{kind}_{year}.csv" for kind in KINDS for year in YEARS]
missing = [name for name in wanted if not (DATA_RAW / name).exists()]
print(f"{len(wanted) - len(missing)} of {len(wanted)} files already in data/raw/; {len(missing)} to download")

## Download the missing files from Kaggle

Skipped when every file is already in `data/raw/`. Otherwise this needs the Kaggle token (see the top of the notebook).
It also saves Kaggle's metadata for the dataset — title, owner and **licence** — to `data/raw/dataset-metadata.json` (Requirement 4.15).

In [ ]:
if missing:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()

    for name in missing:
        # unzip=True: Kaggle sends large files zipped; this extracts them and deletes the zip
        api.dataset_download_file(DATASET, name, path=DATA_RAW, quiet=True, unzip=True)
        assert (DATA_RAW / name).exists(), f"download failed: {name}"
        print(f"downloaded: {name}")

    api.dataset_metadata(DATASET, path=DATA_RAW)
else:
    print("Nothing to download.")

In [ ]:
meta_path = DATA_RAW / "dataset-metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    if isinstance(meta, str):  # some versions of the kaggle package store the JSON as a string
        meta = json.loads(meta)
    print("Title:  ", meta.get("title"))
    print("Licence:", meta.get("licenses"))
else:
    print("No metadata file: record the licence from the Kaggle page in data/README.md by hand.")

## Check what is in `data/raw/`

GitHub rejects files over 100 MB, and its website upload accepts at most 25 MB per file. Files above the limit must stay out of the repository; this notebook then serves as the instructions for obtaining them (Requirement 4.15).

In [ ]:
summary = pd.DataFrame(
    [
        {
            "file": name,
            "size_mb": round((DATA_RAW / name).stat().st_size / 1e6, 1),
            "rows": sum(1 for _ in open(DATA_RAW / name, encoding="utf-8", errors="replace")) - 1,
        }
        for name in wanted
    ]
)
summary["under_github_limit"] = summary["size_mb"] < 100
print(f"Total: {summary['size_mb'].sum():.1f} MB, {summary['rows'].sum():,} rows")
summary

In [ ]:
# A first look at one year of each kind
for kind in KINDS:
    df = pd.read_csv(DATA_RAW / f"{kind}_{YEARS[0]}.csv", nrows=5)
    print(f"{kind}_{YEARS[0]}.csv — columns: {list(df.columns)}")
    display(df)